# Chapter 6 — 네트워크 데이터 구조와 Feature (실습)

## 실습 목표
1. scapy 로 PCAP 를 표(DataFrame)로 변환한다.
2. Zeek conn.log 샘플을 pandas 로 분석한다.
3. 페이로드 엔트로피·IAT feature 함수를 구현한다.
4. 라벨별 feature 분포 차이를 확인한다.

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q scapy

In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
import math
import subprocess
from collections import Counter, defaultdict
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scapy.all import rdpcap, IP, TCP, Raw

---
## Step 1. scapy 로 PCAP → 표(DataFrame)

In [ ]:
# 재현성·오프라인 실행을 위해 scapy 로 합성 PCAP 생성 (실제 샘플과 동일 계층: Ether/IP/TCP/UDP/DNS/HTTP).
from scapy.all import Ether, IP, TCP, UDP, DNS, DNSQR, DNSRR, Raw, wrpcap
from scapy.layers.http import HTTPRequest

PCAP_PATH = Path.cwd() / "http_sample.pcap"

if not PCAP_PATH.exists():
    packets = [
        # DNS 쿼리 + 응답
        Ether()/IP(src="192.168.1.10", dst="8.8.8.8")/UDP(sport=54321, dport=53)/DNS(rd=1, qd=DNSQR(qname="example.com")),
        Ether()/IP(src="8.8.8.8", dst="192.168.1.10")/UDP(sport=53, dport=54321)/DNS(qr=1, qd=DNSQR(qname="example.com"), an=DNSRR(rrname="example.com", rdata="93.184.216.34")),
        Ether()/IP(src="192.168.1.10", dst="8.8.8.8")/UDP(sport=54322, dport=53)/DNS(rd=1, qd=DNSQR(qname="www.google.com")),
        Ether()/IP(src="8.8.8.8", dst="192.168.1.10")/UDP(sport=53, dport=54322)/DNS(qr=1, qd=DNSQR(qname="www.google.com"), an=DNSRR(rrname="www.google.com", rdata="172.217.27.4")),
        # TCP 3-way handshake → example.com:80
        Ether()/IP(src="192.168.1.10", dst="93.184.216.34")/TCP(sport=50432, dport=80, flags="S", seq=1000),
        Ether()/IP(src="93.184.216.34", dst="192.168.1.10")/TCP(sport=80, dport=50432, flags="SA", seq=2000, ack=1001),
        Ether()/IP(src="192.168.1.10", dst="93.184.216.34")/TCP(sport=50432, dport=80, flags="A", seq=1001, ack=2001),
        # HTTP GET 3건 (다양한 User-Agent)
        Ether()/IP(src="192.168.1.10", dst="93.184.216.34")/TCP(sport=50432, dport=80, flags="PA", seq=1001, ack=2001)/HTTPRequest(Method=b"GET", Path=b"/", Host=b"example.com", User_Agent=b"Mozilla/5.0 (Windows NT 10.0; Win64; x64) Firefox/120.0"),
        Ether()/IP(src="192.168.1.10", dst="93.184.216.34")/TCP(sport=50433, dport=80, flags="PA", seq=3000, ack=2100)/HTTPRequest(Method=b"GET", Path=b"/index.html", Host=b"example.com", User_Agent=b"curl/8.0"),
        Ether()/IP(src="192.168.1.10", dst="172.217.27.4")/TCP(sport=50434, dport=80, flags="PA", seq=5000, ack=4000)/HTTPRequest(Method=b"GET", Path=b"/search?q=test", Host=b"www.google.com", User_Agent=b"Mozilla/5.0 (X11; Linux) Chrome/118"),
        # 일반 TCP 데이터 패킷들
        Ether()/IP(src="93.184.216.34", dst="192.168.1.10")/TCP(sport=80, dport=50432, flags="PA")/Raw(load=b"HTTP/1.1 200 OK\r\nContent-Type: text/html\r\n\r\n<html><body>Example</body></html>"),
        Ether()/IP(src="93.184.216.34", dst="192.168.1.10")/TCP(sport=80, dport=50432, flags="FA", seq=2500, ack=1100),
        Ether()/IP(src="192.168.1.10", dst="93.184.216.34")/TCP(sport=50432, dport=80, flags="FA", seq=1100, ack=2501),
    ]
    wrpcap(str(PCAP_PATH), packets)
    print(f"합성 PCAP 생성: {PCAP_PATH} ({len(packets)} packets)")
else:
    print(f"기존: {PCAP_PATH}")


In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. scapy 로 PCAP → 표(DataFrame)
# 만들 결과: pkts, t0, recs, ip, df_pkt
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# scapy 로 핵심 필드를 추출해 DataFrame 구성 (tshark 불필요 — Colab/로컬 공통)
from scapy.all import rdpcap, IP, TCP
pkts = ...
t0 = ...
recs = ...
for p in pkts:
    ip = ...
    ...
df_pkt = ...
...
```
</details>


---
## Step 2. 합성 Zeek conn.log 분석

실제 Zeek 설치 대신, 위의 PCAP 에서 flow 단위로 conn.log 형식을 모사한다.

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 합성 Zeek conn.log 분석 — 실제 Zeek 설치 대신, 위의 PCAP 에서 flow 단위로 conn.log 형식을 모사한다.
# 만들 결과: to_conn_log(), df, df['key'], rows, ts, conn
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def to_conn_log(df_pkt: ...
    df = ...
    df['key'] = ...
    rows = ...
    for key, g in df.groupby('key'):
        ts = ...
        ...
    return ...

conn = ...
...
conn.sort_values('orig_bytes', ascending = ...
```
</details>


---
## Step 3. 페이로드 엔트로피

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 페이로드 엔트로피
# 만들 결과: payload_entropy(), counts, total, entropies
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def payload_entropy(payload: ...
    if not payload:
        return ...
    counts = ...
    total = ...
    return ...

# 본 PCAP 의 모든 TCP 페이로드 엔트로피 분포
entropies = ...
for p in rdpcap(str(PCAP_PATH)):
    if TCP in p and Raw in p:
        ...

if entropies:
    fig, ax = ...
    ax.hist(entropies, bins = ...
    ax.axvline(7.0, color = ...
    ...
    print(f'평균 entropy = ...
```
</details>


관찰: 일반 HTTP 평문이라 엔트로피는 대부분 ~5 부근. 임계 7.0 을 넘는 것이 거의 없다.
이후 TLS 트래픽을 분석하면 페이로드가 모두 7.5+ 로 몰린다.

---
## Step 4. IAT (Inter-Arrival Time) feature

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. IAT (Inter-Arrival Time) feature
# 만들 결과: iat_features(), diffs, iat_mean, iat_std, iat_max, iat_min
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def iat_features(timestamps: ...
    if len(timestamps) < 2:
        return ...
    diffs = ...
    return ...
        iat_mean = ...
        iat_std = ...
        iat_max = ...
        iat_min = ...
    ...

# flow 별 IAT 계산
flow_iat = ...
for p in rdpcap(str(PCAP_PATH)):
    if IP not in p:
        continue
    key = ...
    ...

iat_rows = ...
iat_df = ...
...
```
</details>


---
## Step 5. Feature 정리 — 본 챕터에서 만든 feature 한눈에

In [ ]:
summary = pd.DataFrame([
    {'feature': 'orig_bytes', 'category': '통계', 'source': 'conn'},
    {'feature': 'orig_pkts', 'category': '통계', 'source': 'conn'},
    {'feature': 'duration', 'category': '시간', 'source': 'conn'},
    {'feature': 'payload_entropy', 'category': '페이로드', 'source': 'PCAP'},
    {'feature': 'iat_mean/std/max/min', 'category': '시간', 'source': 'PCAP'},
])
summary

## 보완 실습. SYN·ARP·DNS 관찰 구간 집계

본문 6.4.4절과 같은 합성 패킷을 사용합니다. 패킷은 전송하지 않습니다. 기존 PCAP에도 적용한 뒤 정상 DNS 응답 변경과 공격 의심 응답을 구별할 추가 근거를 정리합니다.

**확인 과제**

1. 두 DNS 응답 주소를 같게 바꾸면 충돌 수가 어떻게 바뀌는지 확인합니다.
2. 응답 주소는 다르게 두고 질의 ID를 다르게 바꿔 비교합니다.
3. ARP 응답 수만으로 스푸핑을 확정할 수 없는 이유와 추가로 확인할 IP–MAC 대응을 정리합니다.

In [ ]:
from collections import defaultdict
from scapy.all import Ether, IP, TCP, UDP, ARP, DNS, DNSQR, DNSRR

def summarize_window(packets):
    syn = tcp = arp_replies = 0
    answers = defaultdict(set)
    times = []
    for packet in packets:
        times.append(float(packet.time))
        if TCP in packet:
            tcp += 1
            flags = int(packet[TCP].flags)
            syn += int(bool(flags & 2) and not bool(flags & 16))
        if ARP in packet:
            arp_replies += int(packet[ARP].op == 2)
        if (IP in packet and UDP in packet and DNS in packet
                and packet[DNS].qr == 1 and DNSQR in packet
                and DNSRR in packet and packet[DNSRR].type == 1):
            key = (packet[IP].dst, packet[UDP].dport,
                   packet[DNS].id, bytes(packet[DNSQR].qname).lower())
            answers[key].add(str(packet[DNSRR].rdata))
    duration = max(times) - min(times) if len(times) > 1 else 0.0
    return {'packets': len(times), 'duration_s': duration,
            'syn_ratio': syn / tcp if tcp else 0.0,
            'arp_replies': arp_replies,
            'dns_answer_conflicts': sum(len(values) > 1 for values in answers.values())}

# 전송하지 않는 합성 패킷입니다. 타임스탬프도 직접 지정합니다.
fixture = [Ether()/IP()/TCP(flags='S'), Ether()/IP()/TCP(flags='A'),
           Ether()/ARP(op=2, hwsrc='02:00:00:00:00:03')]
for answer in ['192.0.2.10', '192.0.2.20']:
    fixture.append(Ether()/IP(src='192.0.2.53', dst='192.0.2.1')/
                   UDP(sport=53, dport=53000)/
                   DNS(id=42, qr=1, qd=DNSQR(qname='example.test'),
                       an=DNSRR(rrname='example.test', type='A', rdata=answer)))
for index, packet in enumerate(fixture):
    packet.time = float(index)
window = summarize_window(fixture)
assert window['syn_ratio'] == 0.5 and window['arp_replies'] == 1
assert window['dns_answer_conflicts'] == 1
print(pd.DataFrame([window]).to_string(index=False))


---
## 정리

- scapy 로 PCAP 파싱 → pandas 분석은 표준 워크플로우다.
- Zeek conn.log 의 통계 컬럼만으로도 flow 단위 ML feature 가 된다.
- 페이로드 엔트로피·IAT 같은 고급 feature 는 단순 통계로는 보이지 않는 패턴을 잡는다.
- 다음 챕터(ch07) 부터는 본격 머신러닝 단계 — 데이터 분석 방법론을 적용한다.